# ECMWF ENS — Probability Analysis

**Model:** ECMWF Ensemble Forecast System — 51 members  
**Station:** TGPY — Maurice Bishop International Airport, Grenada

Reads the pre-computed ECMWF ENS probability products and displays
exceedance probabilities for precipitation and wave height at TGPY.

In [ ]:
import cfgrib
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

TGPY_LAT =  12.0042
TGPY_LON = -61.7862

# ── Set this to the run you want to analyse ──────────────────────────────────
RUN_DIR = Path("data") / sorted((Path("data")).iterdir())[-1].name
# ─────────────────────────────────────────────────────────────────────────────

print(f"Run: {RUN_DIR.name}")
print(f"Exists: {RUN_DIR.exists()}")

In [ ]:
def nearest_point(ds):
    return ds.sel(latitude=TGPY_LAT, longitude=TGPY_LON, method="nearest", tolerance=0.5)

def load_prob_grib(filepath, var_names):
    """Load probability GRIB fields at TGPY. Returns dict {var: [values]}."""
    result = {}
    for ds in cfgrib.open_datasets(str(filepath)):
        pt = nearest_point(ds)
        for var in pt.data_vars:
            if any(v in var for v in var_names):
                vals = pt[var].values
                result[var] = vals.flatten().tolist() if vals.ndim > 0 else [float(vals)]
    return result

In [ ]:
# ── Precipitation probability ─────────────────────────────────────────────────
precip_file = RUN_DIR / "ens_prob_precip_d1-d5.grib2"
precip = load_prob_grib(precip_file, ["tpg1", "tpg5"])

windows = ["D1 (0-24h)", "D2 (24-48h)", "D3 (48-72h)", "D4 (72-96h)", "D5 (96-120h)"]
print("ECMWF ENS — Precipitation Probability at TGPY")
print("-" * 50)
print(f"{'Window':>14}  {'P(>1mm/day)':>12}  {'P(>5mm/day)':>12}")
print("-" * 50)
for i, w in enumerate(windows):
    p1 = precip.get("tpg1", [None]*5)[i]
    p5 = precip.get("tpg5", [None]*5)[i]
    p1s = f"{float(p1):.0%}" if p1 is not None else "—"
    p5s = f"{float(p5):.0%}" if p5 is not None else "—"
    print(f"  {w:>12}  {p1s:>12}  {p5s:>12}")

In [ ]:
# ── Wave height probability ───────────────────────────────────────────────────
wave_file = RUN_DIR / "ens_prob_wave_d1-d5.grib2"
wave = load_prob_grib(wave_file, ["swhg2", "swhg3", "swhg4"])

wave_steps = [f"T+{h:03d}h" for h in range(12, 121, 12)]
print("\nECMWF ENS — Wave Height Probability at TGPY")
print("-" * 55)
print(f"{'Step':>8}  {'P(SWH>2m)':>10}  {'P(SWH>3m)':>10}  {'P(SWH>4m)':>10}")
print("-" * 55)
for i, step in enumerate(wave_steps):
    g2 = wave.get("swhg2", [None]*10)[i]
    g3 = wave.get("swhg3", [None]*10)[i]
    g4 = wave.get("swhg4", [None]*10)[i]
    fmt = lambda v: f"{float(v):.0%}" if v is not None else "—"
    print(f"  {step:>6}  {fmt(g2):>10}  {fmt(g3):>10}  {fmt(g4):>10}")